# Lecture 8 — Qubitization & Quantum Signal Processing (UnitaryLab companion)

Quantum signal processing (QSP) interleaves a fixed *signal* rotation (the
$W$-gate carrying the eigenphase) with tunable $Z$-rotations to realize a
degree-$d$ polynomial transformation of the signal $x = \cos\theta$. Combined
with **qubitization** — which turns a block encoding into a walk operator whose
eigenphases encode the eigenvalues of $A$ — QSP applies a polynomial $p(A)$ to a
state. The canonical application is Hamiltonian simulation: choosing the QSP phases
so that $p(x) \approx \cos(t x)$ (real part) realizes $e^{-\mathrm{i} t H}$ via the
Jacobi–Anger expansion.

`ua.QSPAlgorithm().run(t, d, x)` builds a degree-$d$ QSP sequence targeting
$\cos(t x)$ and evaluates it at a signal value $x$. This notebook verifies the
achieved transform against the target at several sample points and shows the
degree-vs-accuracy tradeoff.

Contents:
1. Setup
2. Introspect the QSP result dictionary
3. Accuracy vs. polynomial degree
4. Matching $\cos(t x)$ across sample points
5. Your turn

In [1]:
import numpy as np
import unitarylab_algorithms as ua
from unitarylab.core import Circuit
import contextlib, io

# Suppress the algorithms' verbose logging so cell outputs stay clean.
@contextlib.contextmanager
def quiet():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yield

np.set_printoptions(precision=4, suppress=True)
print("ready: unitarylab_algorithms loaded")

ready: unitarylab_algorithms loaded


## 2. Introspect the QSP result dictionary

The algorithm targets $\cos(t x)$. It returns the achieved (complex) value, the
ideal value $\cos(t x)$, and their absolute error. We inspect these before
asserting.

In [2]:
with quiet():
    res = ua.QSPAlgorithm().run(1.0, 12, x=0.5, backend="numpy")
print("keys:", list(res.keys()))
print("Estimated value :", np.round(res["Estimated value"], 6))
print("Ideal value     :", round(float(res["Ideal value"]), 6), " (= cos(t*x) = cos(0.5))")
print("Absolute error  :", format(res["Absolute error"], ".2e"))

keys: ['status', 'circuit_path', 'plot', 'circuit', 'Estimated value', 'Ideal value', 'Absolute error', 'Computation time (s)']
Estimated value : (0.872704-0.002738j)
Ideal value     : 0.877583  (= cos(t*x) = cos(0.5))
Absolute error  : 5.59e-03


## 3. Accuracy vs. polynomial degree

The Jacobi–Anger series for $\cos(tx)$ converges rapidly once the degree exceeds
$\sim e|t|/2$. We sweep the degree at fixed $t=1$, $x=0.5$ and watch the error
fall. A low degree ($d=2$) is a poor fit; by $d=12$ the error is well below
$10^{-3}$.

In [3]:
t, x = 1.0, 0.5
ideal = np.cos(t * x)
print(f"target cos(t*x) = {ideal:.6f}\n")
print(" d    estimated   abs error")
errs = {}
for d in [2, 4, 8, 12]:
    with quiet():
        r = ua.QSPAlgorithm().run(t, d, x=x, backend="numpy")
    est = r["Estimated value"].real
    errs[d] = r["Absolute error"]
    print(f"{d:2d}   {est:9.6f}   {r['Absolute error']:.2e}")

# Convergence: degree 12 must be far more accurate than degree 2.
assert errs[12] < errs[2]
assert errs[12] < 5e-3
print("\nVerified: degree-12 QSP approximates cos(t*x) to < 5e-3 (method-appropriate tol).")

target cos(t*x) = 0.877583

 d    estimated   abs error
 2    1.000000   1.22e-01
 4    1.000000   1.22e-01
 8    0.885290   7.71e-03
12    0.873982   3.81e-03

Verified: degree-12 QSP approximates cos(t*x) to < 5e-3 (method-appropriate tol).


## 4. Matching $\cos(t x)$ across sample points

A single point could match by accident, so we evaluate the degree-12 sequence at
several signal values $x \in [-1, 1]$ and check each against $\cos(t x)$. QSP
polynomials are guaranteed $|p(x)| \le 1$ on this interval, which is why the
target is a *bounded* function like cosine.

In [4]:
t, d = 1.0, 12
xs = [-0.8, -0.4, 0.0, 0.3, 0.7]
print("   x      QSP Re      cos(t x)     |err|")
max_err = 0.0
for x in xs:
    with quiet():
        r = ua.QSPAlgorithm().run(t, d, x=x, backend="numpy")
    est = r["Estimated value"].real
    tgt = np.cos(t * x)
    e = abs(est - tgt)
    max_err = max(max_err, e)
    print(f"{x:5.1f}   {est:9.6f}   {tgt:9.6f}   {e:.2e}")

assert max_err < 5e-3
print(f"\nVerified: degree-12 QSP tracks cos(t x) across sample points (max err {max_err:.2e} < 5e-3).")

   x      QSP Re      cos(t x)     |err|
 -0.8    0.696268    0.696707   4.39e-04
 -0.4    0.923166    0.921061   2.11e-03
  0.0    0.999999    1.000000   5.97e-07
  0.3    0.956367    0.955336   1.03e-03
  0.7    0.767588    0.764842   2.75e-03

Verified: degree-12 QSP tracks cos(t x) across sample points (max err 2.75e-03 < 5e-3).


## 5. Your turn
- Increase $t$ to $2$ or $3$ and find the degree needed to hold the error below
  $10^{-3}$ — confirm it grows roughly linearly in $t$.
- Plot QSP estimate vs. $\cos(tx)$ over a dense grid of $x$ (use `matplotlib` with
  the `Agg` backend and `plt.show()`).
- Read off why bounded target functions are required: a QSP polynomial obeys
  $|p(x)| \le 1$ on $[-1, 1]$.

See Lecture 8 for the qubitization walk operator and how its $\pm\arccos(\lambda)$
eigenphases feed QSP to realize $p(A)$.